# PyCaret SNR Classification Workflow

## What to run
Use the **"Clean Voltage-Group PyCaret Workflow"** section near the end of this notebook as the primary pipeline.

## Goal
Compare model performance across different voltage groups and identify which setup performs best.

## Notes
- Need to run create_snr_elem.py first
- Earlier sections are kept as experiment history.
- The clean section is modular and easier to re-run when data updates.
- Recommended environment: `550_model` (Python 3.9 + PyCaret).

In [ ]:
import pandas as pd

df_snr_elem = pd.read_csv(r'C:\Users\phuynh\Projects\robotray-main\models_phan\df_snr_elem.csv')

df_snr_elem
# print all unique target labels
print("Unique target labels:", df_snr_elem['target'].unique())

### GROUP VOLTAGES IN DIFFERENT WAYS

In [4]:
### GROUP BY MODE
 
df0 = df_snr_elem.copy()

mining_voltages = ['mininghighvoltage', 'mininglowvoltage']
soil_voltages   = ['soilhighvoltage', 'soilmidvoltage', 'soillowvoltage']

meta_cols = ['test', 'target', 'voltage']
feature_cols = [c for c in df0.columns if c not in meta_cols]


def build_group(df_in, volt_list, new_voltage_label):
    d = df_in[df_in['voltage'].isin(volt_list)].copy()

    # Pivot so each original voltage becomes a column suffix
    wide = d.pivot_table(
        index=['test', 'target'],
        columns='voltage',
        values=feature_cols,
        aggfunc='first'
    )

    # Flatten MultiIndex columns
    wide.columns = [f"{feat}_{volt}" for feat, volt in wide.columns]

    wide = wide.reset_index()
    wide['voltage'] = new_voltage_label

    return wide


# Build mining and soil datasets
df_mining = build_group(df0, mining_voltages, 'mining')
df_soil   = build_group(df0, soil_voltages, 'soil')

# Stack them vertically → 2 rows per test
df_snr_elem_2v = pd.concat([df_mining, df_soil], ignore_index=True)

print("Original rows:", df_snr_elem.shape[0])
print("New rows (should be 2 × #tests):", df_snr_elem_2v.shape[0])
print("New columns:", df_snr_elem_2v.shape[1])
print(df_snr_elem_2v[['voltage', 'test', 'target']].head())

for col in df_snr_elem_2v.columns:
    print(col)

Original rows: 29213
New rows (should be 2 × #tests): 11684
New columns: 778
  voltage                                 test  target
0  mining  017762_2026_04_01_rutile_150_017762  rutile
1  mining  017762_2026_04_01_rutile_150_017763  rutile
2  mining  017762_2026_04_01_rutile_150_017764  rutile
3  mining  017762_2026_04_01_rutile_150_017765  rutile
4  mining  017762_2026_04_01_rutile_150_017766  rutile
test
target
Ag_Ka_mininghighvoltage
Ag_Ka_mininglowvoltage
Ag_Kb_mininghighvoltage
Ag_Kb_mininglowvoltage
Ag_La_mininghighvoltage
Ag_La_mininglowvoltage
Ag_Lb_mininghighvoltage
Ag_Lb_mininglowvoltage
Al_Ka_mininghighvoltage
Al_Ka_mininglowvoltage
Al_Kb_mininghighvoltage
Al_Kb_mininglowvoltage
As_Ka_mininghighvoltage
As_Ka_mininglowvoltage
As_Kb_mininghighvoltage
As_Kb_mininglowvoltage
Au_La_mininghighvoltage
Au_La_mininglowvoltage
Au_Lb_mininghighvoltage
Au_Lb_mininglowvoltage
Ba_Ka_mininghighvoltage
Ba_Ka_mininglowvoltage
Ba_La_mininghighvoltage
Ba_La_mininglowvoltage
Ba_Lb_mininghighv

In [5]:
# --- GROUP BY MODE (NO NaNs): mining rows have ONLY mining columns; soil rows have ONLY soil columns ---

df0 = df_snr_elem.copy()

mining_voltages = ['mininghighvoltage', 'mininglowvoltage']
soil_voltages   = ['soilhighvoltage', 'soilmidvoltage', 'soillowvoltage']

meta_cols = ['test', 'target', 'voltage']
feature_cols = [c for c in df0.columns if c not in meta_cols]


def build_group_no_nan(df_in: pd.DataFrame, volt_list: list[str], new_voltage_label: str) -> pd.DataFrame:
    d = df_in[df_in['voltage'].isin(volt_list)].copy()

    wide = d.pivot_table(
        index=['test', 'target'],
        columns='voltage',
        values=feature_cols,
        aggfunc='first'
    )

    # Flatten MultiIndex columns -> <feature>_<original_voltage>
    wide.columns = [f"{feat}_{volt}" for feat, volt in wide.columns]
    wide = wide.reset_index()

    # Add mode label
    wide.insert(0, 'voltage', new_voltage_label)

    # IMPORTANT: keep ONLY columns belonging to this volt_list (plus meta)
    keep_cols = ['voltage', 'test', 'target'] + [
        c for c in wide.columns
        if any(c.endswith(f"_{v}") for v in volt_list)
    ]
    wide = wide[keep_cols]

    return wide


df_mining = build_group_no_nan(df0, mining_voltages, 'mining')
df_soil   = build_group_no_nan(df0, soil_voltages,   'soil')

# Stack vertically -> 2 rows per test, with different feature sets per row type
df_snr_elem_2v = pd.concat([df_mining, df_soil], ignore_index=True, sort=False)

print("Original rows:", df_snr_elem.shape[0])
print("New rows (should be 2 × #tests):", df_snr_elem_2v.shape[0])
print("Mining columns:", df_mining.shape[1])
print("Soil columns:", df_soil.shape[1])
print("Combined columns:", df_snr_elem_2v.shape[1])

print(df_snr_elem_2v[['voltage', 'test', 'target']].head())

# Verify: there should be NO NaNs introduced by mixing modes (missing only if raw data missing)
print("Total NaNs:", df_snr_elem_2v.isna().sum().sum())

Original rows: 29213
New rows (should be 2 × #tests): 11684
Mining columns: 313
Soil columns: 468
Combined columns: 778
  voltage                                 test  target
0  mining  017762_2026_04_01_rutile_150_017762  rutile
1  mining  017762_2026_04_01_rutile_150_017763  rutile
2  mining  017762_2026_04_01_rutile_150_017764  rutile
3  mining  017762_2026_04_01_rutile_150_017765  rutile
4  mining  017762_2026_04_01_rutile_150_017766  rutile
Total NaNs: 4527085


In [6]:
### GROUP BY MINERAL

df0 = df_snr_elem.copy()

meta_cols = ['test', 'target', 'voltage']
feature_cols = [c for c in df0.columns if c not in meta_cols]

# Pivot: index = (test, target), columns = voltage, values = feature_cols
wide = df0.pivot_table(
    index=['test', 'target'],
    columns='voltage',
    values=feature_cols,
    aggfunc='first'
)

# Flatten MultiIndex columns -> <feature>_<voltage>
wide.columns = [f"{feat}_{volt}" for feat, volt in wide.columns]

df_snr_elem_1row = wide.reset_index()  # one row per (test, target)

print("Original rows:", df_snr_elem.shape[0])
print("New rows (should be #tests):", df_snr_elem_1row.shape[0])
print("New columns:", df_snr_elem_1row.shape[1])
print(df_snr_elem_1row.head())

for col in df_snr_elem_1row.columns:
    print(col)

# Optional export
# df_snr_elem_1row.to_csv(r'C:\Users\phuynh\Projects\robotray-main\models_phan\df_snr_elem_1row.csv', index=False)
# df_snr_elem_1row.to_parquet(r'C:\Users\phuynh\Projects\robotray-main\models_phan\df_snr_elem_1row.parquet', index=False)

Original rows: 29213
New rows (should be #tests): 5849
New columns: 777
                                  test  target  Ag_Ka_mininghighvoltage  \
0  017762_2026_04_01_rutile_150_017762  rutile                -0.071651   
1  017762_2026_04_01_rutile_150_017763  rutile                 0.163576   
2  017762_2026_04_01_rutile_150_017764  rutile                -0.538349   
3  017762_2026_04_01_rutile_150_017765  rutile                -0.537161   
4  017762_2026_04_01_rutile_150_017766  rutile                -0.094661   

   Ag_Ka_mininglowvoltage  Ag_Ka_soilhighvoltage  Ag_Ka_soillowvoltage  \
0                     0.0              -0.650994              0.001081   
1                     0.0               0.000650              0.349788   
2                     0.0               0.001908              0.000033   
3                     0.0              -0.650994             -0.603556   
4                     0.0               0.374879             -0.603556   

   Ag_Ka_soilmidvoltage  Ag_Kb_m

In [11]:
# Count of empty or NaN cells in a DataFrame
nan_count = df_snr_elem.isna().sum().sum()
print(f"df_snr_elem: Total empty or NaN cells: {nan_count}")

# Count of empty or NaN cells in a DataFrame
nan_count = df_snr_elem_2v.isna().sum().sum()
print(f"df_snr_elem_2v: Total empty or NaN cells: {nan_count}")

# Count of empty or NaN cells in a DataFrame
nan_count = df_mining.isna().sum().sum()
print(f"df_mining: Total empty or NaN cells: {nan_count}")

# Count of empty or NaN cells in a DataFrame
nan_count = df_soil.isna().sum().sum()
print(f"df_soil: Total empty or NaN cells: {nan_count}")

# Count of empty or NaN cells in a DataFrame
nan_count = df_snr_elem_1row.isna().sum().sum()
print(f"df_snr_elem_1row: Total empty or NaN cells: {nan_count}")


df_snr_elem: Total empty or NaN cells: 0
df_snr_elem_2v: Total empty or NaN cells: 4527085
df_mining: Total empty or NaN cells: 155
df_soil: Total empty or NaN cells: 155
df_snr_elem_1row: Total empty or NaN cells: 4960


### ANALYSIS: 1 Voltage 

In [8]:
df_snr_elem

,voltage,test,target,Mg_Ka,Mg_Kb,Al_Ka,Al_Kb,Si_Ka,Si_Kb,P_Ka,...,Ag_Kb,Sn_Ka,Cd_Kb,Sb_Ka,In_Kb,Te_Ka,I_Ka,Sn_Kb,Ba_Ka,Sb_Kb
0,mininghighvoltage,017762_2026_04_01_rutile_150_017762,rutile,0.111330,0.554819,0.551051,0.121553,-0.245867,-0.005592,0.067865,...,0.123584,-0.560609,0.395465,0.174276,0.346040,0.565729,-0.185073,-0.055882,0.125868,0.287064
1,mininghighvoltage,017762_2026_04_01_rutile_150_017763,rutile,-0.528419,-0.002756,-0.307041,0.057917,-0.381054,0.775370,-0.000317,...,1.078962,0.538901,-0.112043,-0.124466,-0.119112,-0.283024,0.424769,0.269064,0.385528,-0.846955
2,mininghighvoltage,017762_2026_04_01_rutile_150_017764,rutile,0.488640,-0.057732,-0.677470,0.585238,-0.651419,-0.006148,-0.403445,...,0.288100,-0.215932,-0.442454,-0.421294,-0.636914,-0.562131,0.105734,-0.014895,0.646154,-0.120952
3,mininghighvoltage,017762_2026_04_01_rutile_150_017765,rutile,-0.056768,0.447379,-0.613286,-0.777158,0.135337,-0.540532,-0.063042,...,-0.214442,0.121390,0.039188,-0.574073,-0.718674,-0.398425,-0.795668,0.902739,-0.522360,-0.260090
4,mininghighvoltage,017762_2026_04_01_rutile_150_017766,rutile,-1.022614,0.872537,0.555321,0.419425,-0.717746,0.782866,-1.141882,...,0.748598,-0.038774,0.153635,-0.376202,0.047529,-0.153091,0.071043,0.074678,-0.810608,-0.023052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29208,soilmidvoltage,024962_2026_04_21_tray_150_025107,tray,0.735390,0.040748,0.231353,0.235091,-0.073781,0.466445,-0.337478,...,-0.756400,-0.387273,0.062975,-0.082308,0.679835,0.456653,-0.480850,0.141757,0.454463,-0.151784
29209,soilmidvoltage,024962_2026_04_21_tray_150_025108,tray,0.454265,-0.729777,-0.770499,0.156374,-1.093554,0.391360,-0.081265,...,-0.659310,-0.252693,-0.028873,0.234274,0.721446,0.190693,-0.530957,-1.033956,0.902371,-0.724663
29210,soilmidvoltage,024962_2026_04_21_tray_150_025109,tray,-0.497105,0.421937,-0.160379,0.537479,-0.242471,0.081259,-0.164166,...,0.434335,0.437339,0.199596,0.010930,0.203159,0.370561,0.185676,0.097385,-0.953258,0.336021
29211,soilmidvoltage,024962_2026_04_21_tray_150_025110,tray,-0.245720,-0.922425,-0.004561,0.307935,-0.161640,0.387304,-0.168833,...,-0.148492,0.427794,0.055688,-0.761298,-0.311806,-0.230192,-0.148584,-0.238655,-0.584107,0.741052


In [9]:
import pandas as pd

df_snr_elem = pd.read_csv(r'C:\Users\phuynh\Projects\robotray-main\models_phan\df_snr_elem.csv')

In [10]:
# ARCHIVED (Legacy cell) - kept for historical comparison only.
# Prefer the Clean Voltage-Group PyCaret Workflow section.

from pycaret.classification import setup, compare_models, predict_model, finalize_model
import pandas as pd

# Select one voltage
voltage_value = df_snr_elem['voltage'].unique()[0]   # or specify manually
df_v = df_snr_elem[df_snr_elem['voltage'] == voltage_value].copy()

print("Training on voltage:", voltage_value)
print("Rows:", df_v.shape[0])

# Keep a copy of test for later (aligned by row order)
test_series = df_v['test'].reset_index(drop=True)

# Drop test BEFORE setup so it cannot be used by the model,
# but we can add it back to predictions later.
df_v_model = df_v.drop(columns=['test', 'voltage']).copy()
# Setup PyCaret
clf = setup(
    data=df_v_model,
    target='target',
    session_id=123,
    use_gpu=False,
    verbose=False
)

# Compare models
best = compare_models()

# Evaluate on holdout set (preds does NOT include test since model data didn't have it)
preds = predict_model(best, raw_score=True)

# Add test back (predict_model preserves row order of the holdout predictions)
# If you want test ONLY for the holdout rows, we need the holdout indices:
holdout_idx = preds.index
preds.insert(0, 'test', test_series.loc[holdout_idx].values)

# Extract per-class probability columns (your pycaret uses prediction_score_<class>)
score_cols = [c for c in preds.columns if c.startswith('prediction_score_')]

probs = preds[score_cols] * 100
probs.columns = [c.replace('prediction_score_', 'Confidence_') for c in score_cols]

final_result = pd.concat([preds.reset_index(drop=True), probs.reset_index(drop=True)], axis=1)

cols_to_show = ['test', 'target', 'prediction_label'] + score_cols
cols_to_show = [c for c in cols_to_show if c in final_result.columns]

print(final_result[cols_to_show].head())
# Optional: train final model on full single-voltage dataset
final_model = finalize_model(best)

ImportError: cannot import name '_Scorer' from 'sklearn.metrics._scorer' (c:\Users\phuynh\AppData\Local\miniconda3\envs\550_model\lib\site-packages\sklearn\metrics\_scorer.py)

In [ ]:
from pycaret.classification import evaluate_model, get_config
classes = sorted(get_config("y").unique())

print("Class mapping:")
for i, c in enumerate(classes):
    print(f"{i}: {c}")

Class mapping:
0: apatite
1: feldspar
2: garnet
3: monazite-aus
4: monazite-bra
5: nephe
6: quartz
7: rutile
8: tray3
9: zircon


In [ ]:
import numpy as np
import pandas as pd
import shap
from pycaret.classification import get_config

# -------------------------
# 1) Ensure preds has 'test'
# -------------------------
preds = preds.copy()
holdout_idx = preds.index
preds["test"] = test_series.loc[holdout_idx].values
preds = preds[["test"] + [c for c in preds.columns if c != "test"]]

# -------------------------
# 2) Prob columns + "how wrong"
# -------------------------
score_cols = [c for c in preds.columns if c.startswith("prediction_score_")]
if len(score_cols) == 0:
    raise ValueError("No 'prediction_score_' columns found. Use predict_model(..., raw_score=True).")

class_to_col = {c.replace("prediction_score_", ""): c for c in score_cols}

def prob_for_label(row, label):
    col = class_to_col.get(str(label))
    return row[col] if col in row else np.nan

preds["pred_prob"] = preds.apply(lambda r: prob_for_label(r, r["prediction_label"]), axis=1)
preds["true_prob"] = preds.apply(lambda r: prob_for_label(r, r["target"]), axis=1)
preds["wrong_margin"] = preds["pred_prob"] - preds["true_prob"]

wrong = preds[preds["target"] != preds["prediction_label"]].copy()
wrong = wrong.sort_values("wrong_margin", ascending=False).copy()

# -------------------------
# 3) Attach SNR values (from df_v)
# -------------------------
snr_cols = [c for c in df_v.columns if c not in ["test", "voltage", "target"]]
snr_holdout = df_v.loc[wrong.index, ["test"] + snr_cols].copy().drop(columns=["test"], errors="ignore")

# -------------------------
# 4) ENERGY MAP (single source of truth)
# -------------------------
energy_map = {f"{el}_{lt}": float(e) for (el, lt, e) in energy_lines}

def energy_keV_for_feature(feat: str) -> float:
    return energy_map.get(str(feat), np.nan)

# -------------------------
# 5) SHAP on WRONG rows (PyCaret-transformed X_test)
# -------------------------
X_test = get_config("X_test")
X_wrong = X_test.loc[wrong.index].copy()
feature_names = list(X_wrong.columns)

# Use TreeExplainer if possible; fallback to generic Explainer
try:
    explainer = shap.TreeExplainer(best)
    shap_values = explainer.shap_values(X_wrong)
except Exception:
    explainer = shap.Explainer(best, X_test)
    exp = explainer(X_wrong)
    shap_values = exp.values

def get_row_shap_vector(row_pos: int, pred_label) -> np.ndarray:
    class_names = list(class_to_col.keys())
    pred_label = str(pred_label)
    class_i = class_names.index(pred_label) if pred_label in class_names else 0

    if isinstance(shap_values, list):
        return np.array(shap_values[class_i][row_pos, :])

    shap_arr = np.array(shap_values)
    if shap_arr.ndim == 3:
        return shap_arr[row_pos, :, class_i]
    return shap_arr[row_pos, :]

# -------------------------
# 6) Flatten top-10 SHAP into ONE ROW per wrong sample
# -------------------------
top_k = 10
flat_rows = []

for row_pos, idx in enumerate(X_wrong.index):
    pred_label = wrong.loc[idx, "prediction_label"]
    shap_vec = get_row_shap_vector(row_pos, pred_label)

    order = np.argsort(np.abs(shap_vec))[::-1][:top_k]

    flat = {}
    for rank_i, j in enumerate(order, start=1):
        feat = feature_names[j]
        flat[f"feature_{rank_i}"]     = str(feat)
        flat[f"energy_keV_{rank_i}"]  = energy_keV_for_feature(feat)
        flat[f"value_{rank_i}"]       = X_wrong.iloc[row_pos, j]
        flat[f"shap_value_{rank_i}"]  = float(shap_vec[j])

    flat_rows.append(pd.Series(flat, name=idx))

shap_flat = pd.DataFrame(flat_rows)

# -------------------------
# 7) Assemble final export (ONE ROW per wrong sample)
# -------------------------
base_cols = ["test", "target", "prediction_label"] + score_cols + ["pred_prob", "true_prob", "wrong_margin"]
export_df = wrong[base_cols].join(snr_holdout, how="left").join(shap_flat, how="left")

# Convert probabilities to percent
for c in score_cols + ["pred_prob", "true_prob", "wrong_margin"]:
    export_df[c] = export_df[c] * 100

export_df = export_df.sort_values("wrong_margin", ascending=False)

# -------------------------
# 8) Export CSV (Excel-readable)
# -------------------------
out_path = "wrong_predictions.csv"
export_df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print("Wrong samples:", export_df.shape[0])
print("Columns:", export_df.shape[1])

Saved: wrong_predictions.csv
Wrong samples: 229
Columns: 158


In [ ]:
preds = predict_model(best, raw_score=True)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.7441,0.9570,0.7441,0.7405,0.7398,0.7142,0.7148


In [ ]:
import numpy as np
import pandas as pd

# Assumes you already ran:
# preds = predict_model(best, raw_score=True)
# and you already defined:
# test_series = df_v['test'].reset_index(drop=True)

# -------------------------
# 1) Ensure `test` is present (overwrite if it already exists)
# -------------------------
preds = preds.copy()
holdout_idx = preds.index
preds["test"] = test_series.loc[holdout_idx].values

# Optional: make `test` first column
preds = preds[["test"] + [c for c in preds.columns if c != "test"]]

# -------------------------
# 2) Identify probability columns from PyCaret (multi-class)
# -------------------------
score_cols = [c for c in preds.columns if c.startswith("prediction_score_")]
if len(score_cols) == 0:
    raise ValueError(
        "No 'prediction_score_' columns found. "
        "Run predict_model(best, raw_score=True) and check preds.columns."
    )

# Map class label -> its probability column
class_to_col = {c.replace("prediction_score_", ""): c for c in score_cols}

def prob_for_label(row, label):
    col = class_to_col.get(str(label))
    return row[col] if col in row else np.nan

# Add per-row diagnostics (still in 0-1 units)
preds["pred_prob"] = preds.apply(lambda r: prob_for_label(r, r["prediction_label"]), axis=1)
preds["true_prob"] = preds.apply(lambda r: prob_for_label(r, r["target"]), axis=1)
preds["wrong_margin"] = preds["pred_prob"] - preds["true_prob"]

# -------------------------
# 3) Filter wrong predictions
# -------------------------
wrong = preds[preds["target"] != preds["prediction_label"]].copy()

# -------------------------
# 4) Convert probabilities to percent for readability
# -------------------------
to_percent_cols = score_cols + ["pred_prob", "true_prob", "wrong_margin"]
for c in to_percent_cols:
    if c in wrong.columns:
        wrong[c] = wrong[c] * 100

# -------------------------
# 5) Print most wrong rows (optional)
# -------------------------
base_cols = ["test", "target", "prediction_label", "pred_prob", "true_prob", "wrong_margin"]
show_cols = [c for c in base_cols + score_cols if c in wrong.columns]

print("\nMOST CONFIDENTLY WRONG (top 25):")
print(wrong.sort_values("pred_prob", ascending=False)[show_cols].head(25))

print("\nMOST WRONG BY MARGIN (top 25):")
print(wrong.sort_values("wrong_margin", ascending=False)[show_cols].head(25))

# -------------------------
# 6) Export to Excel
#    Requirement: export shows ALL columns containing 'pred' in the name
# -------------------------
pred_named_cols = [c for c in wrong.columns if "pred" in c.lower()]

# Also keep key identifiers + all probability columns
must_keep = ["test", "target", "prediction_label"] + score_cols
export_cols = []
for c in must_keep + pred_named_cols:
    if c in wrong.columns and c not in export_cols:
        export_cols.append(c)

wrong_export = wrong[export_cols].copy()

# Export wrong predictions as CSV (Excel-readable)

output_path = "wrong_predictions.csv"
wrong_export.to_csv(output_path, index=False)

print(f"Exported: {output_path}")


MOST CONFIDENTLY WRONG (top 25):
                                           test        target  \
1444      004767_2026_02_23_feldspar_150_004878      feldspar   
2578  005967_2026_02_26_monazite-aus_150_006012  monazite-aus   
1341      004767_2026_02_23_feldspar_150_004775      feldspar   
1686         005067_2026_02_25_nephe_150_005120         nephe   
2603  005967_2026_02_26_monazite-aus_150_006037  monazite-aus   
1233  004617_2026_02_23_monazite-aus_150_004667  monazite-aus   
1375      004767_2026_02_23_feldspar_150_004809      feldspar   
1994       005367_2026_02_25_apatite_150_005428       apatite   
2679  005967_2026_02_26_monazite-aus_150_006113  monazite-aus   
2581  005967_2026_02_26_monazite-aus_150_006015  monazite-aus   
1292  004617_2026_02_23_monazite-aus_150_004726  monazite-aus   
2627  005967_2026_02_26_monazite-aus_150_006061  monazite-aus   
1331  004617_2026_02_23_monazite-aus_150_004765  monazite-aus   
2640  005967_2026_02_26_monazite-aus_150_006074  monazit

In [ ]:
snr_all = pd.read_csv(r'C:/Users/phuynh/Projects/robotray-main/plots_phan/cached_spectrums/snr_all.csv')
print(snr_all)

          Energy (keV)  SNR            voltage  \
0            -0.011061  0.0  mininghighvoltage   
1             0.008980  0.0  mininghighvoltage   
2             0.029020  0.0  mininghighvoltage   
3             0.049061  0.0  mininghighvoltage   
4             0.069102  0.0  mininghighvoltage   
...                ...  ...                ...   
30537723     40.931504  0.0     soillowvoltage   
30537724     40.951544  0.0     soillowvoltage   
30537725     40.971585  0.0     soillowvoltage   
30537726     40.991625  0.0     soillowvoltage   
30537727     41.011666  0.0     soillowvoltage   

                                         test  target  
0          003165_2026_02_11_nephe_150_003165   nephe  
1          003165_2026_02_11_nephe_150_003165   nephe  
2          003165_2026_02_11_nephe_150_003165   nephe  
3          003165_2026_02_11_nephe_150_003165   nephe  
4          003165_2026_02_11_nephe_150_003165   nephe  
...                                       ...     ...  
3053772

In [ ]:
# Identify wrong predictions
wrong = preds[preds['target'] != preds['prediction_label']].copy()

# All prediction-related columns (prediction_label + all prediction_score_*)
pred_cols = [c for c in wrong.columns if "pred" in c]

# Compute confidence of predicted class dynamically
def get_pred_conf(row):
    col = f"prediction_score_{row['prediction_label']}"
    return row[col] if col in wrong.columns else None

wrong['pred_conf'] = wrong.apply(get_pred_conf, axis=1)

# Sort by most confident wrong predictions
wrong = wrong.sort_values('pred_conf', ascending=False)

# Build display column list
display_cols = ['test', 'voltage', 'target'] + pred_cols + ['pred_conf']
display_cols = [c for c in display_cols if c in wrong.columns]

#export wrong to csv
wrong.to_csv("wrong_predictions.csv", index=False)

print("Wrong rows:", wrong.shape[0])
print(wrong[display_cols].head(50))



Wrong rows: 229
                                           test        target  \
1444      004767_2026_02_23_feldspar_150_004878      feldspar   
2578  005967_2026_02_26_monazite-aus_150_006012  monazite-aus   
1341      004767_2026_02_23_feldspar_150_004775      feldspar   
1686         005067_2026_02_25_nephe_150_005120         nephe   
2603  005967_2026_02_26_monazite-aus_150_006037  monazite-aus   
1233  004617_2026_02_23_monazite-aus_150_004667  monazite-aus   
1375      004767_2026_02_23_feldspar_150_004809      feldspar   
1994       005367_2026_02_25_apatite_150_005428       apatite   
2679  005967_2026_02_26_monazite-aus_150_006113  monazite-aus   
2581  005967_2026_02_26_monazite-aus_150_006015  monazite-aus   
1292  004617_2026_02_23_monazite-aus_150_004726  monazite-aus   
2627  005967_2026_02_26_monazite-aus_150_006061  monazite-aus   
1331  004617_2026_02_23_monazite-aus_150_004765  monazite-aus   
2640  005967_2026_02_26_monazite-aus_150_006074  monazite-aus   
133      

In [ ]:
pd.crosstab(
    preds['target'],
    preds['prediction_label']
)

prediction_label,apatite,feldspar,garnet,monazite-aus,monazite-bra,nephe,quartz,rutile,tray3,zircon
target,,,,,,,,,,
apatite,63,5,1,1,3,13,0,2,2,0
feldspar,6,57,4,14,2,1,2,0,2,2
garnet,0,4,85,0,1,0,0,0,0,0
monazite-aus,4,26,0,31,11,4,2,1,0,11
monazite-bra,1,9,0,10,67,3,0,0,0,0
nephe,17,0,1,6,1,55,7,0,0,3
quartz,3,1,0,2,0,2,76,0,5,1
rutile,5,2,0,1,0,0,0,122,0,0
tray3,0,1,0,2,2,0,14,0,26,0


In [ ]:
score_cols = [c for c in preds.columns if c.startswith('Score_')]

incorrect[['target', 'prediction_label'] + score_cols].head()

NameError: name 'incorrect' is not defined

### ARCHIVED (Legacy) - Per-voltage ungrouped analysis

Retained for history only. Prefer the **Clean Voltage-Group PyCaret Workflow** section.

In [32]:
from pycaret.classification import setup, compare_models, predict_model, finalize_model, pull

df_model = df_snr_elem.copy()

print("Rows:", df_model.shape[0])
print("Columns:", df_model.shape[1])
print("Voltage categories:")
print(df_model['voltage'].value_counts())
print("Classes:")
print(df_model['target'].value_counts())

clf = setup(
    data=df_model,
    target='target',
    categorical_features=['voltage'],   # important
    session_id=123,
    use_gpu=False,
    verbose=False
)

best = compare_models()

# Get comparison table
results = pull()
print(results)

# Holdout predictions
preds = predict_model(best, raw_score=True)
print(preds.head())

# Train final model on full dataset
final_model = finalize_model(best)

Rows: 14911
Columns: 105
Voltage categories:
voltage
mininghighvoltage    2983
mininglowvoltage     2982
soilhighvoltage      2982
soillowvoltage       2982
soilmidvoltage       2982
Name: count, dtype: int64
Classes:
target
rutile          2165
nephe           1500
zircon          1500
apatite         1500
monazite-aus    1500
feldspar        1500
quartz          1500
monazite-bra    1499
garnet          1497
tray3            750
Name: count, dtype: int64


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.8307,0.9842,0.8307,0.8331,0.8301,0.8107,0.8110,6.2260
dummy,Dummy Classifier,0.1452,0.5000,0.1452,0.0211,0.0368,0.0000,0.0000,0.0480


                                    Model  Accuracy     AUC  Recall   Prec.  \
lightgbm  Light Gradient Boosting Machine    0.8307  0.9842  0.8307  0.8331   
dummy                    Dummy Classifier    0.1452  0.5000  0.1452  0.0211   

              F1   Kappa    MCC  TT (Sec)  
lightgbm  0.8301  0.8107  0.811     6.226  
dummy     0.0368  0.0000  0.000     0.048  


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.8757,0.9913,0.8757,0.8763,0.8748,0.8611,0.8613


                 voltage                                       test  \
1908   mininghighvoltage        005217_2026_02_25_zircon_150_005342   
3643    mininglowvoltage        003998_2026_02_18_rutile_150_004058   
10027     soillowvoltage  004467_2026_02_23_monazite-bra_150_004515   
9670      soillowvoltage        003998_2026_02_18_rutile_150_004138   
9847      soillowvoltage        004317_2026_02_20_garnet_150_004334   

       Mg_K_alpha  Mg_K_beta  Al_K_alpha  Al_K_beta  Si_K_alpha  Si_K_beta  \
1908    -0.635161   0.291122   -0.174776   0.038238    0.065846   0.719398   
3643     0.108960  -0.269764   -0.048504  -0.356047    0.248032   0.023861   
10027    0.601747  -0.001471   -0.000625  -0.605374    0.423824  -0.608089   
9670     0.000366   0.001213   -0.489324   0.813012   -0.269453   0.345741   
9847    -0.605441   0.347705    0.000975   0.808911   -0.272675  -0.609311   

       P_K_alpha  P_K_beta  ...  prediction_score_apatite  \
1908    0.393059  0.135404  ...            

### ARCHIVED (Legacy) - All voltages grouped analysis

Retained for history only. Prefer the **Clean Voltage-Group PyCaret Workflow** section.

In [33]:
from pycaret.classification import setup, compare_models, predict_model, finalize_model, pull

df_model = df_snr_elem_1row.copy()

print("Rows:", df_model.shape[0])
print("Columns:", df_model.shape[1])
print("Classes:", df_model['target'].value_counts())

clf = setup(
    data=df_model,
    target='target',
    ignore_features=[],   # no session, no voltage anymore
    session_id=123,
    use_gpu=False,
    verbose=False
)

best = compare_models()

# View comparison table
results = pull()
print(results)

# Evaluate best model on holdout set
preds = predict_model(best, raw_score=True)
print(preds.head())

# Train final model on full dataset
final_model = finalize_model(best)

Rows: 2983
Columns: 512
Classes: target
rutile          433
nephe           300
zircon          300
apatite         300
garnet          300
monazite-bra    300
monazite-aus    300
feldspar        300
quartz          300
tray3           150
Name: count, dtype: int64


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.8951,0.9932,0.8951,0.9042,0.8943,0.8829,0.8841,0.5510
lr,Logistic Regression,0.7874,0.0000,0.7874,0.7975,0.7889,0.7625,0.7634,0.1630
knn,K Neighbors Classifier,0.7404,0.9322,0.7404,0.7606,0.7428,0.7100,0.7116,0.0980
svm,SVM - Linear Kernel,0.7237,0.0000,0.7237,0.7366,0.7253,0.6911,0.6921,0.1140
et,Extra Trees Classifier,0.7112,0.9574,0.7112,0.7486,0.7005,0.6767,0.6841,0.1580
ridge,Ridge Classifier,0.7064,0.0000,0.7064,0.7064,0.6989,0.6718,0.6735,0.0980
lightgbm,Light Gradient Boosting Machine,0.3401,0.8456,0.3401,0.3460,0.2880,0.2617,0.3598,4.5800
ada,Ada Boost Classifier,0.2328,0.0000,0.2328,0.1492,0.1556,0.1410,0.2715,0.9650
dummy,Dummy Classifier,0.1451,0.5000,0.1451,0.0211,0.0368,0.0000,0.0000,0.3470
dt,Decision Tree Classifier,0.1164,0.5087,0.1164,0.0288,0.0371,0.0170,0.0342,0.1700


                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.8951  0.9932  0.8951  0.9042   
lr                    Logistic Regression    0.7874  0.0000  0.7874  0.7975   
knn                K Neighbors Classifier    0.7404  0.9322  0.7404  0.7606   
svm                   SVM - Linear Kernel    0.7237  0.0000  0.7237  0.7366   
et                 Extra Trees Classifier    0.7112  0.9574  0.7112  0.7486   
ridge                    Ridge Classifier    0.7064  0.0000  0.7064  0.7064   
lightgbm  Light Gradient Boosting Machine    0.3401  0.8456  0.3401  0.3460   
ada                  Ada Boost Classifier    0.2328  0.0000  0.2328  0.1492   
dummy                    Dummy Classifier    0.1451  0.5000  0.1451  0.0211   
dt               Decision Tree Classifier    0.1164  0.5087  0.1164  0.0288   
gbc          Gradient Boosting Classifier    0.1068  0.0000  0.1068  0.0652   
qda       Quadratic Discriminant Analysis    0.1020 

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.8950,0.9918,0.8950,0.8979,0.8935,0.8827,0.8834


                                     test  Ag_K_alpha_mininghighvoltage  \
892   004317_2026_02_20_garnet_150_004326                      0.491587   
2882  006267_2026_02_27_quartz_150_006316                      0.188615   
278    003329_2026_02_12_tray3_150_003457                     -0.114315   
1721   005067_2026_02_25_nephe_150_005155                     -0.268862   
774   004148_2026_02_19_rutile_150_004189                      0.531130   

      Ag_K_alpha_mininglowvoltage  Ag_K_alpha_soilhighvoltage  \
892                           0.0                   -0.002781   
2882                          0.0                   -0.651514   
278                           0.0                    0.650758   
1721                          0.0                    1.060337   
774                           0.0                    0.000022   

      Ag_K_alpha_soillowvoltage  Ag_K_alpha_soilmidvoltage  \
892                   -0.604185                  -0.614368   
2882                  -0.000316   

### ARCHIVED (Legacy) - By mode, two separate PyCaret runs

Retained for history only. Prefer the **Clean Voltage-Group PyCaret Workflow** section. 

In [34]:
from pycaret.classification import setup, compare_models, predict_model, finalize_model, pull

def run_pycaret_one_voltage_group(df_2v, group_label):
    df_model = df_2v[df_2v['voltage'] == group_label].copy()

    # For this subset, drop the now-constant 'voltage' column
    df_model = df_model.drop(columns=['voltage'])

    print("\n=== Running:", group_label, "===")
    print("Rows:", df_model.shape[0], "Cols:", df_model.shape[1])
    print("Classes:\n", df_model['target'].value_counts())

    setup(
        data=df_model,
        target='target',
        session_id=123,
        use_gpu=False,
        verbose=False
    )

    best = compare_models()
    results = pull()

    preds = predict_model(best, raw_score=True)   # holdout predictions
    final_model = finalize_model(best)

    return best, results, preds, final_model

best_mining, results_mining, preds_mining, final_mining = run_pycaret_one_voltage_group(df_snr_elem_2v, 'mining')
best_soil,   results_soil,   preds_soil,   final_soil   = run_pycaret_one_voltage_group(df_snr_elem_2v, 'soil')

print("\n--- Mining model comparison table ---")
print(results_mining)

print("\n--- Soil model comparison table ---")
print(results_soil)


=== Running: mining ===
Rows: 2983 Cols: 512
Classes:
 target
rutile          433
nephe           300
zircon          300
apatite         300
garnet          300
monazite-bra    300
monazite-aus    300
feldspar        300
quartz          300
tray3           150
Name: count, dtype: int64


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.7390,0.9683,0.7390,0.7891,0.7350,0.7080,0.7146,0.2100
lr,Logistic Regression,0.7333,0.0000,0.7333,0.7574,0.7358,0.7019,0.7040,0.2790
ridge,Ridge Classifier,0.7251,0.0000,0.7251,0.7225,0.7126,0.6927,0.6954,0.0600
knn,K Neighbors Classifier,0.7165,0.9152,0.7165,0.7422,0.7212,0.6831,0.6851,0.0610
svm,SVM - Linear Kernel,0.6600,0.0000,0.6600,0.6796,0.6457,0.6202,0.6242,0.0690
et,Extra Trees Classifier,0.6293,0.9395,0.6293,0.7094,0.6094,0.5849,0.5998,0.1070
ada,Ada Boost Classifier,0.2328,0.0000,0.2328,0.1492,0.1556,0.1410,0.2715,0.3870
dummy,Dummy Classifier,0.1451,0.5000,0.1451,0.0211,0.0368,0.0000,0.0000,0.0690
gbc,Gradient Boosting Classifier,0.1307,0.0000,0.1307,0.1253,0.0612,0.0330,0.1160,13.1820
dt,Decision Tree Classifier,0.1154,0.5082,0.1154,0.0282,0.0363,0.0159,0.0322,0.0790


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.7497,0.9685,0.7497,0.8006,0.7442,0.7199,0.7269



=== Running: soil ===
Rows: 2982 Cols: 512
Classes:
 target
rutile          433
nephe           300
zircon          300
apatite         300
monazite-bra    300
monazite-aus    300
feldspar        300
quartz          300
garnet          299
tray3           150
Name: count, dtype: int64


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.8342,0.9894,0.8342,0.8593,0.8302,0.8149,0.8183,0.2930
lr,Logistic Regression,0.6210,0.0000,0.6210,0.6431,0.6231,0.5763,0.5786,0.1470
ridge,Ridge Classifier,0.6095,0.0000,0.6095,0.6021,0.5944,0.5632,0.5660,0.0600
et,Extra Trees Classifier,0.5846,0.9182,0.5846,0.6707,0.5656,0.5347,0.5584,0.1260
svm,SVM - Linear Kernel,0.5794,0.0000,0.5794,0.5908,0.5729,0.5297,0.5328,0.0700
knn,K Neighbors Classifier,0.5611,0.8426,0.5611,0.5956,0.5703,0.5094,0.5118,0.0670
lightgbm,Light Gradient Boosting Machine,0.3584,0.8231,0.3584,0.4361,0.3279,0.2830,0.3891,3.8060
ada,Ada Boost Classifier,0.2065,0.0000,0.2065,0.1224,0.1292,0.1131,0.2181,0.5650
dummy,Dummy Classifier,0.1452,0.5000,0.1452,0.0211,0.0368,0.0000,0.0000,0.0720
gbc,Gradient Boosting Classifier,0.1394,0.0000,0.1394,0.1240,0.0712,0.0404,0.1096,27.2650


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.8547,0.9903,0.8547,0.8704,0.8530,0.8379,0.8401



--- Mining model comparison table ---
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.7390  0.9683  0.7390  0.7891   
lr                    Logistic Regression    0.7333  0.0000  0.7333  0.7574   
ridge                    Ridge Classifier    0.7251  0.0000  0.7251  0.7225   
knn                K Neighbors Classifier    0.7165  0.9152  0.7165  0.7422   
svm                   SVM - Linear Kernel    0.6600  0.0000  0.6600  0.6796   
et                 Extra Trees Classifier    0.6293  0.9395  0.6293  0.7094   
ada                  Ada Boost Classifier    0.2328  0.0000  0.2328  0.1492   
dummy                    Dummy Classifier    0.1451  0.5000  0.1451  0.0211   
gbc          Gradient Boosting Classifier    0.1307  0.0000  0.1307  0.1253   
dt               Decision Tree Classifier    0.1154  0.5082  0.1154  0.0282   
lda          Linear Discriminant Analysis    0.1006  0.0000  0.1006  0.0101   
lightgbm  Lig

### ARCHIVED (Legacy) - Two-dataframe no-NaN variant

Retained for history only. Prefer the **Clean Voltage-Group PyCaret Workflow** section.

In [35]:
from pycaret.classification import setup, compare_models, predict_model, finalize_model, pull

def run_pycaret(df_in, label):
    df_model = df_in.copy()

    # Drop voltage if it exists (constant within each df)
    if 'voltage' in df_model.columns:
        df_model = df_model.drop(columns=['voltage'])

    print(f"\n=== Running: {label} ===")
    print("Rows:", df_model.shape[0], "Cols:", df_model.shape[1])
    print("Classes:\n", df_model['target'].value_counts())

    setup(
        data=df_model,
        target='target',
        session_id=123,
        use_gpu=False,
        verbose=False
    )

    best = compare_models()
    results = pull()

    preds = predict_model(best, raw_score=True)   # holdout predictions
    final_model = finalize_model(best)

    return best, results, preds, final_model


# Use your two separate dataframes (from the no-NaN build step)
best_mining, results_mining, preds_mining, final_mining = run_pycaret(df_mining, "mining")
best_soil,   results_soil,   preds_soil,   final_soil   = run_pycaret(df_soil, "soil")

print("\n--- Mining model comparison table ---")
print(results_mining)

print("\n--- Soil model comparison table ---")
print(results_soil)


=== Running: mining ===
Rows: 2983 Cols: 206
Classes:
 target
rutile          433
nephe           300
zircon          300
apatite         300
garnet          300
monazite-bra    300
monazite-aus    300
feldspar        300
quartz          300
tray3           150
Name: count, dtype: int64


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.7390,0.9683,0.7390,0.7891,0.7350,0.7080,0.7146,0.2480
lr,Logistic Regression,0.7347,0.0000,0.7347,0.7584,0.7370,0.7035,0.7056,0.0960
ridge,Ridge Classifier,0.7251,0.0000,0.7251,0.7225,0.7126,0.6927,0.6954,0.0350
knn,K Neighbors Classifier,0.7165,0.9152,0.7165,0.7422,0.7212,0.6831,0.6851,0.0390
svm,SVM - Linear Kernel,0.6600,0.0000,0.6600,0.6796,0.6457,0.6202,0.6242,0.0410
et,Extra Trees Classifier,0.6293,0.9395,0.6293,0.7094,0.6094,0.5849,0.5998,0.0890
ada,Ada Boost Classifier,0.2328,0.0000,0.2328,0.1492,0.1556,0.1410,0.2715,0.4200
dummy,Dummy Classifier,0.1451,0.5000,0.1451,0.0211,0.0368,0.0000,0.0000,0.0330
gbc,Gradient Boosting Classifier,0.1307,0.0000,0.1307,0.1253,0.0612,0.0330,0.1160,14.3420
dt,Decision Tree Classifier,0.1154,0.5082,0.1154,0.0282,0.0363,0.0159,0.0322,0.0530


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.7497,0.9685,0.7497,0.8006,0.7442,0.7199,0.7269



=== Running: soil ===
Rows: 2982 Cols: 308
Classes:
 target
rutile          433
nephe           300
zircon          300
apatite         300
monazite-bra    300
monazite-aus    300
feldspar        300
quartz          300
garnet          299
tray3           150
Name: count, dtype: int64


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.8342,0.9894,0.8342,0.8593,0.8302,0.8149,0.8183,0.3240
lr,Logistic Regression,0.6210,0.0000,0.6210,0.6431,0.6231,0.5763,0.5786,0.1450
ridge,Ridge Classifier,0.6095,0.0000,0.6095,0.6021,0.5944,0.5632,0.5660,0.0450
et,Extra Trees Classifier,0.5846,0.9182,0.5846,0.6707,0.5656,0.5347,0.5584,0.1050
svm,SVM - Linear Kernel,0.5794,0.0000,0.5794,0.5908,0.5729,0.5297,0.5328,0.0530
knn,K Neighbors Classifier,0.5611,0.8426,0.5611,0.5956,0.5703,0.5094,0.5118,0.0470
lightgbm,Light Gradient Boosting Machine,0.3584,0.8231,0.3584,0.4361,0.3279,0.2830,0.3891,3.3080
ada,Ada Boost Classifier,0.2065,0.0000,0.2065,0.1224,0.1292,0.1131,0.2181,0.5870
dummy,Dummy Classifier,0.1452,0.5000,0.1452,0.0211,0.0368,0.0000,0.0000,0.0420
gbc,Gradient Boosting Classifier,0.1394,0.0000,0.1394,0.1240,0.0712,0.0404,0.1096,22.6990


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.8547,0.9903,0.8547,0.8704,0.8530,0.8379,0.8401



--- Mining model comparison table ---
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.7390  0.9683  0.7390  0.7891   
lr                    Logistic Regression    0.7347  0.0000  0.7347  0.7584   
ridge                    Ridge Classifier    0.7251  0.0000  0.7251  0.7225   
knn                K Neighbors Classifier    0.7165  0.9152  0.7165  0.7422   
svm                   SVM - Linear Kernel    0.6600  0.0000  0.6600  0.6796   
et                 Extra Trees Classifier    0.6293  0.9395  0.6293  0.7094   
ada                  Ada Boost Classifier    0.2328  0.0000  0.2328  0.1492   
dummy                    Dummy Classifier    0.1451  0.5000  0.1451  0.0211   
gbc          Gradient Boosting Classifier    0.1307  0.0000  0.1307  0.1253   
dt               Decision Tree Classifier    0.1154  0.5082  0.1154  0.0282   
lda          Linear Discriminant Analysis    0.1006  0.0000  0.1006  0.0101   
lightgbm  Lig

### ARCHIVED (Legacy) - By mode, single run with voltage feature

Retained for history only. Prefer the **Clean Voltage-Group PyCaret Workflow** section.

In [36]:
from pycaret.classification import setup, compare_models, predict_model, finalize_model, pull

df_model = df_snr_elem_2v.copy()

setup(
    data=df_model,
    target='target',
    categorical_features=['voltage'],
    session_id=123,
    use_gpu=False,
    verbose=False
)

best = compare_models()
results = pull()
preds = predict_model(best, raw_score=True)
final_model = finalize_model(best)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.8901,0.9899,0.8901,0.8950,0.8903,0.8773,0.8778,6.1560
dummy,Dummy Classifier,0.1451,0.5000,0.1451,0.0211,0.0368,0.0000,0.0000,0.0910


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.9006,0.9916,0.9006,0.9038,0.9018,0.8890,0.8891


### ARCHIVED (Legacy) - Voltage as categorical feature

Retained for history only. Prefer the **Clean Voltage-Group PyCaret Workflow** section.

In [37]:
# ARCHIVED (Legacy cell) - kept for historical comparison only.
# Prefer the Clean Voltage-Group PyCaret Workflow section.

from pycaret.classification import setup, compare_models, predict_model, finalize_model
import pandas as pd

# df_snr_elem is your full dataframe (all voltages)
df = df_snr_elem.copy()

# Train unified model (no grouping), ignore metadata/id columns
clf = setup(
    data=df,
    target='target',
    ignore_features=['session', 'test'],
    categorical_features=['voltage'],
    session_id=123,
    use_gpu=False,
    verbose=False
)

best = compare_models()

# Get predictions on PyCaret's holdout set (recommended vs predicting on training data)
preds = predict_model(best, raw_score=True)

# Extract probability columns robustly
nonprob = {'prediction_label', 'prediction_score'}
prob_cols = [c for c in preds.columns if c not in nonprob and c not in df.columns]

probs = preds[prob_cols] * 100
probs.columns = [f'Confidence_{c}' for c in prob_cols]

final_result = pd.concat([preds.reset_index(drop=True), probs.reset_index(drop=True)], axis=1)

# Train final model on full dataset if you want to deploy it
final_model = finalize_model(best)

# final_result contains holdout predictions + confidence columns
print(final_result.head())

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.7044,0.9557,0.7044,0.7136,0.7071,0.6698,0.6702,3.9590
gbc,Gradient Boosting Classifier,0.6605,0.0000,0.6605,0.6660,0.6614,0.6206,0.6210,42.3150
rf,Random Forest Classifier,0.6166,0.9220,0.6166,0.6241,0.6153,0.5713,0.5719,1.3950
dt,Decision Tree Classifier,0.5277,0.7374,0.5277,0.5301,0.5281,0.4727,0.4728,0.3550
lr,Logistic Regression,0.5050,0.0000,0.5050,0.4924,0.4963,0.4464,0.4469,0.0630
et,Extra Trees Classifier,0.4997,0.8487,0.4997,0.5037,0.4890,0.4391,0.4401,0.2720
ridge,Ridge Classifier,0.4726,0.0000,0.4726,0.4421,0.4355,0.4091,0.4133,0.0440
svm,SVM - Linear Kernel,0.4578,0.0000,0.4578,0.4371,0.4402,0.3931,0.3951,0.1370
ada,Ada Boost Classifier,0.4497,0.0000,0.4497,0.4391,0.4374,0.3842,0.3857,1.1640
lda,Linear Discriminant Analysis,0.4442,0.0000,0.4442,0.5249,0.4670,0.3811,0.3834,0.0800


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.7224,0.9603,0.7224,0.7274,0.7241,0.6898,0.6899


             voltage  Mg_K_alpha  Mg_K_beta  Al_K_alpha  Al_K_beta  \
0  mininghighvoltage   -0.635161   0.291122   -0.174776   0.038238   
1   mininglowvoltage    0.108960  -0.269764   -0.048504  -0.356047   
2     soillowvoltage    0.601747  -0.001471   -0.000625  -0.605374   
3     soillowvoltage    0.000366   0.001213   -0.489324   0.813012   
4     soillowvoltage   -0.605441   0.347705    0.000975   0.808911   

   Si_K_alpha  Si_K_beta  P_K_alpha  P_K_beta  S_K_alpha  ...  \
0    0.065846   0.719398   0.393059  0.135404   0.904336  ...   
1    0.248032   0.023861   0.069314  0.678426   0.490897  ...   
2    0.423824  -0.608089  -0.002626 -0.852948  -0.351975  ...   
3   -0.269453   0.345741   0.001174 -0.852814   0.271167  ...   
4   -0.272675  -0.609311   0.268080  0.496191   0.267215  ...   

   Confidence_prediction_score_apatite  Confidence_prediction_score_feldspar  \
0                                 0.09                                  0.10   
1                           

In [38]:
# ARCHIVED (Legacy cell) - kept for historical comparison only.
# Prefer the Clean Voltage-Group PyCaret Workflow section.

from pycaret.classification import setup, compare_models, predict_model, finalize_model, get_config
import pandas as pd

results = []
first_voltage = df_snr_elem['voltage'].unique()[0]
df_filtered = df_snr_elem[df_snr_elem['voltage'] == first_voltage].copy()

clf1 = setup(
    data=df_filtered,
    target='target',
    session_id=123,
    fold_strategy='groupkfold',
    fold=5,
    fold_groups='session',          # change if your true group key is different
    ignore_features=['test', 'session'],  # if session string leaks the label
    verbose=False
)

best = compare_models()

# evaluate on holdout (not training)
preds = predict_model(best, raw_score=True)  # holdout by default

# Extract probability columns robustly
nonprob = {'prediction_label', 'prediction_score'}
prob_cols = [c for c in preds.columns if c not in nonprob and c not in df_filtered.columns]

probs = preds[prob_cols] * 100
probs.columns = [f'Confidence_{c}' for c in prob_cols]

# Combine with holdout rows only
X_holdout = get_config('X_test').reset_index(drop=True)
y_holdout = get_config('y_test').reset_index(drop=True)
base = pd.concat([X_holdout, y_holdout], axis=1)

result = pd.concat([base.reset_index(drop=True), probs.reset_index(drop=True)], axis=1)
result['voltage'] = first_voltage
results.append(result)

final_result = pd.concat(results, ignore_index=True)

ValueError: Invalid value for the fold_groups parameter. Column session is not present in the dataset.

In [ ]:
# Optional: feature importance plot for the best model from a specific group.
# Example:
# from pycaret.classification import plot_model
# plot_model(all_results["mininghighvoltage"]["best_model"], plot="feature")

NameError: name 'plot_model' is not defined

## Clean Voltage-Group PyCaret Workflow

This section is the recommended end-to-end pipeline for your objective:
1. Start from `df_snr_elem` (already built or loaded).
2. Run one PyCaret experiment per voltage group.
3. Compare best model metrics across groups.

> If `df_snr_elem` is not in memory yet, run the earlier data-building/loading cells first.

In [ ]:
from pycaret.classification import setup, compare_models, pull, predict_model, finalize_model
import pandas as pd


def run_pycaret_for_voltage_group(df_all: pd.DataFrame, voltage_value: str):
    """Run one PyCaret experiment for a single voltage subset."""
    df_group = df_all[df_all["voltage"] == voltage_value].copy()

    # Keep metadata for traceability, but do not use it as model features.
    ignore_cols = [c for c in ["test", "session", "voltage"] if c in df_group.columns]

    print(f"\n=== Voltage group: {voltage_value} ===")
    print(f"Rows: {len(df_group):,} | Columns: {df_group.shape[1]}")
    print(df_group["target"].value_counts())

    setup(
        data=df_group,
        target="target",
        ignore_features=ignore_cols,
        session_id=123,
        use_gpu=False,
        verbose=False,
    )

    best_model = compare_models()
    leaderboard = pull().copy()
    holdout_preds = predict_model(best_model, raw_score=True)
    final_model = finalize_model(best_model)

    best_row = leaderboard.iloc[0].copy()
    best_row["voltage_group"] = voltage_value

    return {
        "voltage_group": voltage_value,
        "best_model": best_model,
        "leaderboard": leaderboard,
        "holdout_predictions": holdout_preds,
        "final_model": final_model,
        "best_metrics": best_row,
    }


def compare_all_voltage_groups(df_all: pd.DataFrame):
    """Run and summarize PyCaret results for each voltage group."""
    if "voltage" not in df_all.columns:
        raise ValueError("Expected column 'voltage' in df_snr_elem.")
    if "target" not in df_all.columns:
        raise ValueError("Expected column 'target' in df_snr_elem.")

    groups = sorted(df_all["voltage"].dropna().unique().tolist())
    results = {}
    best_rows = []

    for g in groups:
        out = run_pycaret_for_voltage_group(df_all, g)
        results[g] = out
        best_rows.append(out["best_metrics"])

    summary = pd.DataFrame(best_rows)
    metric_cols = [c for c in ["Accuracy", "AUC", "Recall", "Prec.", "F1", "Kappa", "MCC"] if c in summary.columns]
    summary = summary[["voltage_group", "Model"] + metric_cols]
    summary = summary.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

    return results, summary

In [ ]:
# Run voltage-group experiments
all_results, voltage_summary = compare_all_voltage_groups(df_snr_elem)

print("\n=== Best model by voltage group (sorted by Accuracy) ===")
voltage_summary

# Optional: inspect full leaderboard for one group
# all_results["mininghighvoltage"]["leaderboard"]